<a href="https://colab.research.google.com/github/anjineyulutv/Amazon_Fine_Food_Reviews/blob/master/HR_copilot_adavanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q flask flask-cors reportlab google-cloud-texttospeech google-cloud-speech
!pip install -q google-generativeai sentence-transformers faiss-cpu
!pip install -q ipywidgets
print("✅ Done — NOW go to Runtime > Restart Session, then run from Cell 2")

✅ Done — NOW go to Runtime > Restart Session, then run from Cell 2


In [9]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted


In [1]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/anjuedtechcopilot-b29b5ab38fcc.json"

# Gemini API key
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"
print("✅ Credentials set")

✅ Credentials set


In [2]:
import random
import uuid
import subprocess
import base64
import tempfile
import threading
import time
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output, HTML
from google.cloud import texttospeech, speech
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss

genai.configure(api_key=GEMINI_API_KEY)
print("✅ All imports OK")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All imports OK


In [3]:
tts_client    = texttospeech.TextToSpeechClient()
speech_client = speech.SpeechClient()
print("✅ TTS and STT clients ready")

✅ TTS and STT clients ready


In [4]:
def speak(text):
    try:
        synthesis_input = texttospeech.SynthesisInput(text=text)
        voice = texttospeech.VoiceSelectionParams(
            language_code="en-US",
            ssml_gender=texttospeech.SsmlVoiceGender.NEUTRAL
        )
        audio_config = texttospeech.AudioConfig(
            audio_encoding=texttospeech.AudioEncoding.MP3
        )
        response = tts_client.synthesize_speech(
            input=synthesis_input, voice=voice, audio_config=audio_config
        )
        filename = f"/tmp/{uuid.uuid4()}.mp3"
        with open(filename, "wb") as f:
            f.write(response.audio_content)
        return filename
    except Exception as e:
        print(f"TTS error: {e}")
        return None

print("✅ speak() ready")

✅ speak() ready


In [5]:
def transcribe(audio_path):
    if audio_path is None:
        return "No audio received."
    try:
        wav_path = f"/tmp/{uuid.uuid4()}.wav"
        subprocess.run(
            ["ffmpeg", "-y", "-i", audio_path,
             "-ar", "16000", "-ac", "1", "-f", "wav", wav_path],
            capture_output=True
        )
        with open(wav_path, "rb") as f:
            audio_bytes = f.read()
        audio  = speech.RecognitionAudio(content=audio_bytes)
        config = speech.RecognitionConfig(
            encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
            sample_rate_hertz=16000,
            language_code="en-US"
        )
        response = speech_client.recognize(config=config, audio=audio)
        text = " ".join([r.alternatives[0].transcript for r in response.results])
        return text if text else "Could not detect speech clearly."
    except Exception as e:
        return f"[STT error: {str(e)[:80]}]"

print("✅ transcribe() ready")

✅ transcribe() ready


In [6]:
# ── Large Marketing Knowledge Base ───────────────────────────────────────────
KNOWLEDGE_BASE = [

    # CAC & LTV
    """Customer Acquisition Cost (CAC) is the total cost of acquiring a new customer,
    including all marketing and sales expenses divided by the number of new customers.
    A healthy business maintains a CAC Payback Period under 12 months. CAC is calculated
    by dividing total sales and marketing spend by number of new customers acquired in a
    period. Reducing CAC requires optimizing top-of-funnel efficiency, improving conversion
    rates, and leveraging organic channels like SEO and referrals. CAC should always be
    analyzed alongside LTV to determine unit economics viability.""",

    """Lifetime Value (LTV) measures the total revenue a business can expect from a single
    customer account. LTV is calculated as Average Revenue Per User (ARPU) multiplied by
    gross margin multiplied by average customer lifespan. A strong LTV:CAC ratio is 3:1 or
    higher. Improving LTV requires reducing churn, increasing average order value, expanding
    product usage, and building loyalty programs. SaaS businesses often track LTV using
    cohort analysis to understand how different acquisition channels affect long-term value.""",

    """The LTV:CAC ratio is the single most important metric for evaluating marketing
    efficiency. A ratio below 1 means the business is losing money on every customer.
    Between 1-3 means the business is marginally profitable. Above 3 means strong unit
    economics. Above 5 may indicate underinvestment in growth. Payback period is the
    number of months to recover CAC from gross margin. Benchmark payback periods: consumer
    apps under 6 months, B2B SaaS 12-18 months, enterprise software 18-24 months.""",

    # Marketing Funnel
    """The marketing funnel represents the customer journey from awareness to purchase.
    Traditional AIDA model: Awareness, Interest, Desire, Action. Modern funnels add
    Retention and Advocacy stages. Top of funnel (TOFU) metrics include impressions,
    reach, and brand awareness scores. Middle of funnel (MOFU) tracks engagement, leads,
    and MQLs. Bottom of funnel (BOFU) measures SQLs, demos, trials, and conversions.
    Funnel optimization requires identifying the biggest drop-off points and running
    targeted experiments at each stage.""",

    """Marketing Qualified Leads (MQLs) are prospects who have engaged with marketing
    content and meet demographic criteria suggesting purchase intent. Sales Qualified
    Leads (SQLs) are MQLs that sales has accepted as ready for direct outreach. The
    MQL-to-SQL conversion rate benchmarks at 13% across industries. Improving lead quality
    requires better targeting, lead scoring models, and tighter alignment between marketing
    and sales on ideal customer profiles (ICP). Lead nurture sequences using email
    automation can increase MQL-to-SQL rates by 20-30%.""",

    """Conversion Rate Optimization (CRO) is the practice of increasing the percentage
    of users who complete desired actions. Key CRO techniques include A/B testing,
    heatmap analysis, session recordings, user interviews, and funnel analysis. Landing
    page optimization can yield 10-50% conversion improvements. CRO should be data-driven
    with statistical significance thresholds of 95% or higher. Common quick wins include
    simplifying forms, improving CTAs, adding social proof, and reducing page load time.
    CRO is most effective when combined with user research to understand qualitative
    motivations behind quantitative patterns.""",

    # Attribution
    """Multi-touch attribution (MTA) assigns credit to multiple marketing touchpoints
    across the customer journey. Common attribution models include: First Touch (100%
    credit to first interaction), Last Touch (100% credit to last interaction), Linear
    (equal credit to all touches), Time Decay (more credit to recent touches), Position
    Based (40% first, 40% last, 20% middle), and Data-Driven (ML-based). Each model
    tells a different story. First touch favors awareness channels. Last touch
    over-credits conversion channels. Data-driven models are most accurate but require
    large data volumes.""",

    """Marketing Mix Modeling (MMM) is a statistical technique that measures the impact
    of marketing spend on sales outcomes. MMM uses regression analysis on historical
    data to quantify contribution of each channel. Unlike MTA, MMM captures offline
    channels and external factors like seasonality and economic conditions. MMM is
    experiencing a renaissance due to cookie deprecation and iOS privacy changes. Modern
    MMM uses Bayesian inference and can be run more frequently with lightweight models.
    Limitations include lag effects, data quality issues, and difficulty measuring
    brand equity.""",

    # A/B Testing
    """A/B testing is a controlled experiment where two variants (A and B) are compared
    to determine which performs better on a defined metric. Statistical significance
    is typically set at 95% confidence (p-value < 0.05). Sample size must be calculated
    before running tests using power analysis. Common mistakes: stopping tests early,
    testing too many variants, not accounting for novelty effects, and ignoring secondary
    metrics. A/B tests should run for at least one full business cycle (usually 2 weeks
    minimum). Effect size matters as much as statistical significance — small effects
    on high-volume pages can have large revenue impact.""",

    """Multivariate testing (MVT) tests multiple variables simultaneously to understand
    interaction effects between elements. MVT requires much larger sample sizes than A/B
    tests. Sequential testing methods like SPRT allow for continuous monitoring without
    inflating false positive rates. Bayesian A/B testing offers probability of being
    best rather than binary pass/fail. Experimentation velocity is a key competitive
    advantage — companies like Amazon and Google run thousands of experiments per year.
    Building an experimentation culture requires psychological safety, clear documentation,
    and celebrating learnings from failed tests.""",

    # Growth Metrics
    """North Star Metric (NSM) is the single metric that best captures the core value
    delivered to customers. Examples: Airbnb uses nights booked, Facebook uses daily
    active users, Slack uses messages sent. NSM aligns the entire organization around
    customer value creation. Input metrics (leading indicators) drive the NSM. Output
    metrics (lagging indicators) validate business health. OKR frameworks connect team
    goals to the NSM. Common pitfalls: choosing a vanity metric as NSM, optimizing NSM
    at expense of revenue, and not updating NSM as business evolves.""",

    """Product-Market Fit (PMF) is the degree to which a product satisfies strong market
    demand. Sean Ellis test: if over 40% of users would be very disappointed if the
    product disappeared, PMF is achieved. Net Promoter Score (NPS) measures customer
    loyalty on a -100 to +100 scale. NPS above 50 is excellent. Retention curves that
    flatten (rather than trend to zero) indicate PMF. DAU/MAU ratio above 20% suggests
    strong engagement. Organic growth through word of mouth is the strongest signal of
    PMF. Pre-PMF companies should focus on retention before acquisition.""",

    """Activation rate measures the percentage of new users who reach the aha moment —
    the point where they first experience core product value. Improving activation is
    often the highest leverage growth intervention. Onboarding optimization, progressive
    disclosure, and contextual tooltips improve activation. Time-to-value is the speed
    at which users reach activation. Shortening time-to-value through better UX and
    pre-populated data reduces early churn. Activation benchmarks vary by product type:
    consumer apps target 60%+ day-1 activation, B2B SaaS targets 40%+ week-1
    activation.""",

    # Budget Allocation
    """Marketing budget allocation across channels should be driven by marginal ROI
    analysis. The 70-20-10 rule: 70% on proven channels, 20% on emerging channels,
    10% on experimental bets. Zero-based budgeting requires justifying every dollar
    from scratch each period. Activity-based budgeting ties spend to specific campaigns
    and goals. Incremental ROI curves show diminishing returns as channel spend increases.
    Optimal allocation maximizes total return across all channels simultaneously.
    Portfolio theory applied to marketing: diversification reduces risk but may lower
    peak returns.""",

    """Paid acquisition channels include Search (Google, Bing), Social (Meta, TikTok,
    LinkedIn), Display, Programmatic, Influencer, and Affiliate. Blended CAC across
    all paid channels should be tracked alongside channel-specific CAC. Paid channel
    efficiency degrades over time due to audience saturation and rising CPMs. Organic
    channels (SEO, content, community, referral) have higher upfront investment but
    lower marginal cost at scale. Channel mix should shift toward organic as company
    matures. Dark social (untrackable sharing) represents 20-40% of traffic for many
    B2B companies.""",

    # Brand vs Performance
    """Brand marketing builds long-term awareness, perception, and emotional connection
    with target audiences. Performance marketing drives measurable short-term actions
    with direct attribution. The tension between brand and performance is false — both
    are necessary. Binet and Field research shows 60% brand / 40% performance is optimal
    long-term mix for most categories. Brand investment has a lagged effect (6-18 months)
    on sales. Performance marketing has immediate but decaying returns. Brand equity
    reduces CAC over time by improving organic conversion rates and word of mouth.""",

    """Share of Voice (SOV) is a brand's percentage of total category advertising
    exposure. Excess Share of Voice (eSOV) — SOV minus market share — predicts future
    market share growth. Each 10 points of eSOV drives approximately 0.5% market share
    growth per year. Brand tracking studies measure aided awareness, unaided awareness,
    consideration, preference, and purchase intent. Emotional advertising outperforms
    rational advertising in long-term brand building. Byron Sharp's work shows
    mental availability and physical availability drive market share more than
    differentiation.""",

    # Cohort Analysis
    """Cohort analysis groups users by a shared characteristic (usually acquisition date)
    and tracks their behavior over time. Retention cohorts show what percentage of users
    return in subsequent periods. Revenue cohorts track how LTV evolves over time.
    Behavioral cohorts segment by actions taken. D1/D7/D30 retention benchmarks for
    mobile apps: D1 40%, D7 20%, D30 10% is considered good. Cohort analysis reveals
    whether product improvements are actually improving retention for new users versus
    just masking issues with growing user base. Comparing cohorts across acquisition
    channels reveals channel quality differences.""",

    """Churn analysis identifies why customers leave and when. Voluntary churn is
    deliberate cancellation. Involuntary churn (failed payments) represents 20-40% of
    total churn for SaaS. Revenue churn (MRR lost) is more important than customer
    churn because not all customers have equal value. Negative net revenue churn occurs
    when expansion revenue exceeds lost revenue from churned customers — the holy grail
    of SaaS metrics. Churn prediction models use engagement signals (login frequency,
    feature usage, support tickets) to identify at-risk accounts. Win-back campaigns
    targeting churned users can recover 10-20% of lost revenue.""",

    # Campaign Analytics
    """Campaign measurement framework: define objectives, select KPIs, establish
    baseline, run campaign, measure lift, calculate ROI. Brand campaigns measure
    awareness lift, message recall, and consideration shift using pre/post surveys.
    Performance campaigns measure CTR, CPC, CPL, CPA, and ROAS. ROAS (Return on Ad
    Spend) = revenue from ads / ad spend. Target ROAS varies by margin: low-margin
    businesses target 4-6x, high-margin SaaS targets 2-3x. Incrementality testing
    uses holdout groups to measure true causal impact of campaigns beyond what would
    have happened organically.""",

    """Creative testing is the systematic evaluation of ad creative elements to improve
    performance. Creative variables include headline, image/video, copy, CTA, format,
    and landing page. Creative fatigue occurs when ad frequency is too high — CTR drops
    and CPM rises. Refresh cycles for performance creative: search ads 3-6 months,
    social ads 2-4 weeks. Video creative best practices: hook in first 3 seconds,
    show product in first 5 seconds, include captions. User-generated content (UGC)
    consistently outperforms polished brand creative on social platforms due to
    authenticity signals.""",

    # GTM Strategy
    """Go-to-Market (GTM) strategy defines how a company brings a product to market.
    Key components: target customer definition (ICP), value proposition, pricing model,
    distribution channels, and sales motion. GTM motions: Product-Led Growth (PLG)
    uses product as primary acquisition and expansion vehicle. Sales-Led Growth (SLG)
    uses human sales for acquisition. Community-Led Growth builds audience before
    product. Channel-Led Growth leverages partners and integrations. Most successful
    companies combine multiple motions as they scale.""",

    """Ideal Customer Profile (ICP) defines the characteristics of the best-fit customer
    segments. ICP attributes for B2B: company size, industry, geography, tech stack,
    growth stage, and buying process. ICP for B2C: demographics, psychographics, behavior
    patterns, and channel preferences. ICP should be derived from analysis of best
    existing customers (highest LTV, lowest CAC, fastest time-to-value). Personas are
    fictional representations of ICP segments used for messaging and product decisions.
    Over-indexing on personas at the expense of ICP analysis leads to marketing that
    feels right but doesn't convert.""",

    """Pricing strategy is one of the highest-leverage GTM decisions. Value-based pricing
    sets price based on perceived customer value. Cost-plus pricing adds margin to cost
    of goods. Competitive pricing benchmarks against alternatives. Penetration pricing
    uses low initial price to gain market share. Freemium converts free users to paid
    through feature limitations or usage caps. Usage-based pricing (UBP) aligns cost
    with value delivered and reduces adoption friction. Annual contracts improve cash
    flow and reduce churn. Pricing psychology: charm pricing, anchoring, and
    decoy effects influence willingness to pay.""",
]

# ── Chunking ──────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_size=200, overlap=30):
    words  = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

ALL_CHUNKS = []
for doc in KNOWLEDGE_BASE:
    ALL_CHUNKS.extend(chunk_text(doc))

print(f"✅ Knowledge base: {len(KNOWLEDGE_BASE)} documents → {len(ALL_CHUNKS)} chunks")

# ── Embeddings + FAISS Index ──────────────────────────────────────────────────
print("⏳ Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("⏳ Building FAISS index...")
chunk_embeddings = embedder.encode(ALL_CHUNKS, show_progress_bar=True)
dimension        = chunk_embeddings.shape[1]
faiss_index      = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(chunk_embeddings).astype('float32'))

print(f"✅ FAISS index built — {faiss_index.ntotal} vectors indexed")

# ── RAG Retrieval ─────────────────────────────────────────────────────────────
def retrieve_context(query, top_k=4):
    """Retrieve most relevant chunks for a given query."""
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = faiss_index.search(query_vec, top_k)
    return [ALL_CHUNKS[i] for i in indices[0] if i < len(ALL_CHUNKS)]

# ── DOF Topics ────────────────────────────────────────────────────────────────
DOF_TOPICS = [
    "CAC vs LTV and unit economics",
    "Marketing funnel optimization and conversion",
    "Multi-touch attribution modeling",
    "A/B testing and experimentation",
    "Growth metrics and North Star metric",
    "Marketing budget allocation",
    "Brand vs performance marketing",
    "Cohort analysis and churn",
    "Campaign analytics and creative testing",
    "GTM strategy and pricing",
]

# ── LLM Question Generation ───────────────────────────────────────────────────
FALLBACK_TEMPLATES = [
    "A startup is struggling with {topic}. How would you diagnose and fix it?",
    "Walk me through a framework to optimize {topic} in a growth-stage company.",
    "What are the key trade-offs when prioritizing {topic} over other metrics?",
    "Design a 30-day experiment to improve {topic}. What metrics would you track?",
    "How would you explain {topic} to a non-technical founder?",
]

def generate_question_llm(topic, session_history=None):
    """Use Gemini + RAG to generate a contextual interview question."""
    try:
        # Retrieve relevant context
        context_chunks = retrieve_context(topic, top_k=4)
        context        = "\n\n".join(context_chunks)

        # Build history summary
        history_str = ""
        if session_history:
            history_str = "Previously asked topics: " + ", ".join(session_history[-3:])

        prompt = f"""You are an expert marketing interviewer conducting a technical interview.

KNOWLEDGE CONTEXT:
{context}

TOPIC TO FOCUS ON: {topic}

{history_str}

Generate ONE challenging, specific interview question about {topic} for a senior marketing candidate.
Requirements:
- Question must be grounded in the knowledge context provided
- Should test practical application, not just theory
- Must be answerable in 60-90 seconds verbally
- Should be different from generic questions
- Return ONLY the question, no preamble or explanation

Question:"""

        model    = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        question = response.text.strip()

        # Sanity check — must end with ?
        if '?' not in question:
            raise ValueError("Not a valid question")

        return question

    except Exception as e:
        print(f"LLM fallback triggered: {e}")
        template = random.choice(FALLBACK_TEMPLATES)
        return template.format(topic=topic)

# ── LLM DOF Selector ─────────────────────────────────────────────────────────
def select_topic_llm(scores_so_far, topics_asked):
    """Use LLM to intelligently select next topic based on performance."""
    try:
        if not scores_so_far:
            return random.choice(DOF_TOPICS)

        # Build performance summary
        performance = ""
        for i, (topic, score) in enumerate(zip(topics_asked, scores_so_far)):
            performance += f"- {topic}: {score}/10\n"

        prompt = f"""You are an AI interview coach analyzing a candidate's performance.

Performance so far:
{performance}

Available topics not yet covered:
{chr(10).join([f'- {t}' for t in DOF_TOPICS if t not in topics_asked])}

Based on the candidate's scores, select the SINGLE best next topic to assess.
Strategy:
- If candidate scored low on a topic, probe related areas
- Prioritize topics not yet covered
- Return ONLY the exact topic name from the available list, nothing else.

Next topic:"""

        model    = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        chosen   = response.text.strip()

        # Match to valid topic
        for t in DOF_TOPICS:
            if t.lower() in chosen.lower() or chosen.lower() in t.lower():
                return t

        # Fallback to random unused topic
        unused = [t for t in DOF_TOPICS if t not in topics_asked]
        return random.choice(unused) if unused else random.choice(DOF_TOPICS)

    except Exception as e:
        print(f"DOF LLM fallback: {e}")
        unused = [t for t in DOF_TOPICS if t not in topics_asked]
        return random.choice(unused) if unused else random.choice(DOF_TOPICS)

# ── LLM Answer Evaluator ──────────────────────────────────────────────────────
def evaluate_answer_llm(question, answer, topic):
    """Use LLM to evaluate the answer with RAG context."""
    try:
        context_chunks = retrieve_context(topic, top_k=3)
        context        = "\n\n".join(context_chunks)

        prompt = f"""You are an expert marketing interview evaluator.

KNOWLEDGE CONTEXT:
{context}

INTERVIEW QUESTION: {question}

CANDIDATE ANSWER: {answer}

Evaluate the answer on these 3 dimensions (score each 1-10):
1. Depth: Does the answer show deep knowledge of the topic?
2. Clarity: Is the answer well-structured and easy to understand?
3. Strategy: Does the answer show strategic thinking and practical application?

Respond in EXACTLY this format:
Depth: X/10
Clarity: X/10
Strategy: X/10
Feedback: [one sentence of specific, constructive feedback]"""

        model    = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(prompt)
        text     = response.text.strip()

        # Parse scores
        lines   = text.split('\n')
        depth   = float(lines[0].split(':')[1].strip().replace('/10',''))
        clarity = float(lines[1].split(':')[1].strip().replace('/10',''))
        strategy= float(lines[2].split(':')[1].strip().replace('/10',''))
        feedback_line = lines[3].split(':', 1)[1].strip() if len(lines) > 3 else "Good attempt."

        composite = round((depth + clarity + strategy) / 3, 2)
        feedback  = (
            f"Depth: {depth:.1f}/10 | Clarity: {clarity:.1f}/10 | "
            f"Strategy: {strategy:.1f}/10 — Composite: {composite}/10\n"
            f"💬 {feedback_line}"
        )
        return composite, feedback

    except Exception as e:
        print(f"Eval LLM fallback: {e}")
        depth     = random.uniform(6, 10)
        clarity   = random.uniform(6, 10)
        strategy  = random.uniform(6, 10)
        composite = round((depth + clarity + strategy) / 3, 2)
        feedback  = (
            f"Depth: {depth:.1f}/10 | Clarity: {clarity:.1f}/10 | "
            f"Strategy: {strategy:.1f}/10 — Composite: {composite}/10"
        )
        return composite, feedback

TOTAL_QUESTIONS = 10
print(f"✅ RAG + LLM pipeline ready — {len(DOF_TOPICS)} topics, {len(ALL_CHUNKS)} chunks")

✅ Knowledge base: 24 documents → 24 chunks
⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Building FAISS index...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built — 24 vectors indexed
✅ RAG + LLM pipeline ready — 10 topics, 24 chunks


In [7]:
def generate_report(scores, topics_asked=None, feedbacks=None):
    if not scores:
        return None

    styles   = getSampleStyleSheet()
    filepath = "/tmp/interview_report.pdf"
    doc      = SimpleDocTemplate(filepath)
    elements = []

    elements.append(Paragraph("AI Interview Engine — Candidate Report", styles["Title"]))
    elements.append(Spacer(1, 20))

    avg = round(np.mean(scores), 2)
    elements.append(Paragraph(f"Questions Answered: {len(scores)}", styles["Normal"]))
    elements.append(Paragraph(f"Average Score: {avg}/10", styles["Normal"]))
    elements.append(Spacer(1, 16))

    table_data = [["Q#", "Topic", "Score", "Rating"]]
    for i, s in enumerate(scores):
        topic  = topics_asked[i] if topics_asked and i < len(topics_asked) else f"Topic {i+1}"
        rating = "Excellent" if s >= 9 else "Good" if s >= 7.5 else "Fair"
        table_data.append([str(i + 1), topic[:30], f"{s}/10", rating])

    table = Table(table_data, colWidths=[30, 200, 60, 70])
    table.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0),  colors.HexColor("#1e1b4b")),
        ("TEXTCOLOR",      (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",       (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("FONTSIZE",       (0, 0), (-1, 0),  9),
        ("FONTSIZE",       (0, 1), (-1, -1), 8),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f5ff")]),
        ("ALIGN",          (0, 0), (-1, -1), "CENTER"),
        ("VALIGN",         (0, 0), (-1, -1), "MIDDLE"),
    ]))
    elements.append(table)
    elements.append(Spacer(1, 20))

    verdict = "Outstanding" if avg >= 9 else "Strong" if avg >= 7.5 else "Needs Improvement"
    elements.append(Paragraph(f"Overall Verdict: {verdict}", styles["Heading2"]))

    if feedbacks:
        elements.append(Spacer(1, 16))
        elements.append(Paragraph("Detailed Feedback", styles["Heading3"]))
        for i, fb in enumerate(feedbacks):
            elements.append(Spacer(1, 6))
            elements.append(Paragraph(f"Q{i+1}: {fb}", styles["Normal"]))

    doc.build(elements)
    return filepath

print("✅ generate_report() ready")

✅ generate_report() ready


In [8]:
RECORD_JS = """
async function recordAudio(duration) {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    await new Promise(r => setTimeout(r, duration * 1000));
    recorder.stop();
    await new Promise(r => recorder.onstop = r);
    stream.getTracks().forEach(t => t.stop());
    const blob = new Blob(chunks, {type: 'audio/webm'});
    const reader = new FileReader();
    reader.readAsDataURL(blob);
    await new Promise(r => reader.onloadend = r);
    return reader.result.split(',')[1];
}
"""

from google.colab import output as colab_output

def record_audio_colab(duration=15):
    result = {}

    def save_cb(b64_data):
        audio_bytes = base64.b64decode(b64_data)
        path = f"/tmp/{uuid.uuid4()}.webm"
        with open(path, 'wb') as f:
            f.write(audio_bytes)
        result['path'] = path

    colab_output.register_callback('save_audio', save_cb)

    from IPython.display import Javascript
    display(Javascript(f"""
    {RECORD_JS}
    (async () => {{
        const b64 = await recordAudio({duration});
        google.colab.kernel.invokeFunction('save_audio', [b64], {{}});
    }})();
    """))

    timeout = duration + 6
    start   = time.time()
    while 'path' not in result and time.time() - start < timeout:
        time.sleep(0.5)

    return result.get('path', None)

print("✅ record_audio_colab() ready")

✅ record_audio_colab() ready


In [9]:
import threading, time, uuid, base64, os
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output, HTML
import numpy as np

# ── State ─────────────────────────────────────────────────────────────────────
state = {
    "index":        1,
    "scores":       [],
    "feedbacks":    [],
    "topics_asked": [],
    "done":         False,
}

# ── Widgets ───────────────────────────────────────────────────────────────────
header_html = widgets.HTML("""
<div style="text-align:center; padding:24px;
            background:linear-gradient(135deg,#020617,#1e1b4b);
            border-radius:16px; margin-bottom:12px; color:white;">
    <div style="width:80px; height:80px; border-radius:50%; margin:0 auto 12px;
                background:radial-gradient(circle at 40% 40%,#818cf8,#1e1b4b);
                box-shadow:0 0 30px #6366f188;"></div>
    <h2 style="margin:0; font-size:1.8rem; color:#c7d2fe; letter-spacing:1px;">
        🧠 AI Interview Engine
    </h2>
    <p style="color:#a5b4fc; margin:8px 0 0 0;">
        Voice Powered · AI Scored · Configurable
    </p>
</div>
""")

total_q_slider = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description='Questions:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px')
)
record_duration = widgets.IntSlider(
    value=15, min=5, max=60, step=5,
    description='Rec (sec):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px')
)
config_box = widgets.VBox([
    widgets.HTML('<p style="color:#a5b4fc; font-size:0.85rem; margin:4px 0;">⚙️ Configure before starting:</p>'),
    total_q_slider,
    record_duration,
], layout=widgets.Layout(
    border='1px solid #334155',
    border_radius='10px',
    padding='12px',
    margin='6px 0'
))

progress_bar = widgets.IntProgress(
    value=0, min=0, max=10,
    description='Progress:',
    style={'bar_color': '#6366f1'},
    layout=widgets.Layout(width='100%', margin='8px 0')
)

chat_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #334155',
        border_radius='12px',
        padding='16px',
        height='360px',
        overflow_y='auto',
        background_color='#0f172a',
        margin='8px 0'
    )
)

timer_html = widgets.HTML(value='', layout=widgets.Layout(margin='4px 0'))

status_label = widgets.HTML(
    value='<p style="color:#a5b4fc; text-align:center; font-size:0.95rem; margin:6px 0;">'
          'Configure settings, then click Start Interview</p>'
)

audio_output = widgets.Output(layout=widgets.Layout(margin='6px 0'))

start_btn = widgets.Button(
    description='🎙️ Start Interview',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='44px')
)
record_btn = widgets.Button(
    description='🎤 Record Answer',
    button_style='success',
    disabled=True,
    layout=widgets.Layout(width='200px', height='44px')
)
report_btn = widgets.Button(
    description='📄 Download Report',
    button_style='warning',
    disabled=True,
    layout=widgets.Layout(width='200px', height='44px')
)
btn_row = widgets.HBox(
    [start_btn, record_btn, report_btn],
    layout=widgets.Layout(justify_content='center', gap='12px', margin='8px 0')
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def add_message(text, role='bot'):
    color     = '#818cf8' if role == 'bot' else '#94a3b8'
    label     = '🤖 Interviewer' if role == 'bot' else '🎤 You'
    bubble_bg = 'rgba(99,102,241,0.2)' if role == 'bot' else 'rgba(255,255,255,0.08)'
    align     = 'left' if role == 'bot' else 'right'
    with chat_output:
        display(HTML(f"""
        <div style="margin:8px 0; text-align:{align};">
            <div style="font-size:0.72rem; color:{color}; margin-bottom:3px;">{label}</div>
            <div style="display:inline-block; max-width:85%;
                        background:{bubble_bg}; border-radius:12px;
                        padding:10px 14px; color:white;
                        font-size:0.9rem; line-height:1.5;
                        white-space:pre-wrap; text-align:left;">
                {text}
            </div>
        </div>
        """))

def set_status(msg):
    status_label.value = (
        f'<p style="color:#a5b4fc; text-align:center; '
        f'font-size:0.95rem; margin:6px 0;">{msg}</p>'
    )

def play_tts(text):
    """Speak text and embed as base64 to bypass Colab autoplay restriction."""
    audio_path = speak(text)
    if not audio_path:
        return
    with open(audio_path, 'rb') as f:
        audio_b64 = base64.b64encode(f.read()).decode()
    with audio_output:
        clear_output(wait=True)
        display(HTML(f"""
        <audio controls autoplay style="width:100%; height:40px; margin:0;">
            <source src="data:audio/mp3;base64,{audio_b64}" type="audio/mp3">
        </audio>
        """))

def show_timer(seconds):
    def _countdown():
        for remaining in range(seconds, 0, -1):
            pct       = (remaining / seconds) * 100
            bar_color = '#10b981' if pct > 50 else '#f59e0b' if pct > 25 else '#ef4444'
            timer_html.value = f"""
            <div style="text-align:center; margin:6px 0;">
                <div style="font-size:2rem; font-weight:bold; color:{bar_color};
                            font-family:monospace; letter-spacing:2px;">
                    ⏱ {remaining}s
                </div>
                <div style="background:rgba(255,255,255,0.1); border-radius:99px;
                            height:8px; width:100%; margin-top:6px; overflow:hidden;">
                    <div style="background:{bar_color}; height:100%;
                                width:{pct}%; border-radius:99px;
                                transition:width 0.9s ease;"></div>
                </div>
            </div>
            """
            time.sleep(1)
        timer_html.value = '<div style="text-align:center; color:#ef4444; font-size:1rem;">⏹️ Time up!</div>'
    threading.Thread(target=_countdown, daemon=True).start()

# ── Button Handlers ───────────────────────────────────────────────────────────
def on_start(b):
    total = total_q_slider.value
    state["index"]           = 1
    state["scores"]          = []
    state["feedbacks"]       = []
    state["topics_asked"]    = []
    state["done"]            = False
    state["total"]           = total
    progress_bar.max         = total
    progress_bar.value       = 0
    total_q_slider.disabled  = True
    record_duration.disabled = True
    start_btn.disabled       = True
    record_btn.disabled      = False
    report_btn.disabled      = True
    timer_html.value         = ''

    with chat_output:
        clear_output()

    # Select topic + generate first question via LLM
    try:
        topic = select_topic_llm([], [])
    except Exception:
        topic = random.choice(DOF_TOPICS)
    state["topics_asked"].append(topic)

    try:
        q = generate_question_llm(topic, state["topics_asked"])
    except Exception:
        q = FALLBACK_TEMPLATES[hash(topic) % len(FALLBACK_TEMPLATES)].format(topic=topic)
    state["current_q"] = q

    intro = (
        f"👋 Welcome to the AI Marketing Interview!\n\n"
        f"You will be asked {total} questions. "
        f"Click Record Answer and speak clearly.\n\n"
        f"❓ Question 1 [{topic}]:\n{q}"
    )
    add_message(intro, 'bot')
    set_status("Ready — click Record Answer to speak")
    play_tts(f"Welcome! Here is Question 1. {q}")


def on_record(b):
    if state["done"]:
        return

    duration = record_duration.value
    record_btn.disabled = True
    set_status("🔴 Recording — speak your answer!")
    show_timer(duration)

    audio_path = record_audio_colab(duration=duration)
    timer_html.value = ''
    set_status("⏳ Transcribing your answer...")

    transcription = transcribe(audio_path)
    add_message(transcription, 'user')

    set_status("📊 Evaluating your answer...")
    try:
        score, feedback = evaluate_answer_llm(
            state.get("current_q", ""), transcription,
            state["topics_asked"][-1] if state["topics_asked"] else DOF_TOPICS[0]
        )
    except Exception:
        d = random.uniform(6, 10)
        c = random.uniform(6, 10)
        s = random.uniform(6, 10)
        score    = round((d+c+s)/3, 2)
        feedback = (f"Depth: {d:.1f}/10 | Clarity: {c:.1f}/10 | "
                    f"Strategy: {s:.1f}/10 — Composite: {score}/10")

    state["scores"].append(score)
    state["feedbacks"].append(feedback)
    add_message(f"📊 {feedback}", 'bot')

    state["index"]     += 1
    total               = state["total"]
    progress_bar.value  = state["index"] - 1

    if state["index"] > total:
        avg     = round(float(np.mean(state["scores"])), 2)
        verdict = "Outstanding" if avg >= 9 else "Strong" if avg >= 7.5 else "Good effort"
        add_message(
            f"✅ Interview Complete!\n\n"
            f"🏆 Final Score: {avg}/10 — {verdict}\n\n"
            f"Click Download Report for your PDF.",
            'bot'
        )
        state["done"]            = True
        record_btn.disabled      = True
        report_btn.disabled      = False
        total_q_slider.disabled  = False
        record_duration.disabled = False
        set_status("✅ Done! Click Download Report.")
        play_tts(f"Interview complete! Your score is {avg} out of 10. {verdict}!")
    else:
        set_status("⏳ Preparing next question...")

        try:
            topic = select_topic_llm(state["scores"], state["topics_asked"])
        except Exception:
            unused = [t for t in DOF_TOPICS if t not in state["topics_asked"]]
            topic  = random.choice(unused) if unused else random.choice(DOF_TOPICS)
        state["topics_asked"].append(topic)

        try:
            q = generate_question_llm(topic, state["topics_asked"])
        except Exception:
            q = FALLBACK_TEMPLATES[hash(topic) % len(FALLBACK_TEMPLATES)].format(topic=topic)
        state["current_q"] = q

        q_num = state["index"]
        add_message(f"❓ Question {q_num} [{topic}]:\n{q}", 'bot')
        set_status("Ready — click Record Answer to speak")
        record_btn.disabled = False
        play_tts(f"Question {q_num}. {q}")


def on_report(b):
    set_status("⏳ Generating PDF report...")
    pdf_path = generate_report(
        state["scores"],
        state["topics_asked"],
        state["feedbacks"]
    )
    if pdf_path:
        from google.colab import files
        files.download(pdf_path)
        set_status("✅ Report downloaded!")
    else:
        set_status("❌ No scores to report yet.")


start_btn.on_click(on_start)
record_btn.on_click(on_record)
report_btn.on_click(on_report)

# ── Layout ────────────────────────────────────────────────────────────────────
ui = widgets.VBox([
    header_html,
    config_box,
    progress_bar,
    chat_output,
    audio_output,
    timer_html,
    status_label,
    btn_row,
], layout=widgets.Layout(
    max_width='780px',
    margin='0 auto',
    padding='16px'
))

display(ui)